# htr-metrics-transfer: model-aware predictors (H2)

Why do the image-only metrics predict HTR-VT's CER but not TrOCR's? Each recognizer trained **without** augmentation (`cyttic/hmt-<arch>-none-s<seed>`: HTR-VT s42/s43, TrOCR-CTC, TrOCR-small, TrOCR scratch) is used as a probe; nothing is trained.

- **Affinity:** its greedy CER on the dev lines augmented with each condition (3 draws) minus the clean-dev CER.
- **Own-feature metrics:** the same metrics as `metrics_hmt.py` (Cover, Dens, Prec, Vendi, PairD, A_dist, KID), computed in the model's own feature space (the CTC classifier's input frames; `frame` = 6000 sampled frames, `line` = per-line mean), training set augmented with exactly the image-metric draws vs the clean dev lines.

5 probes x 49 jobs, split over the 2 GPUs. Each finished job goes to `cyttic/hmt-metrics` (`model_aware/<probe>/<cond>_k<k>.json`), so **Save & Run All again resumes** after a timeout. The last cell runs the leave-one-family-out test with permutation p-values -> `model_aware/summary.json`. Expected about 2-4 h.

**Setup:** Accelerator = GPU T4 x2, Internet = ON, secret `HF_TOKEN` (write).

In [ ]:
!pip install -q 'pillow==11.2.1' 'transformers==5.15.0' jiwer   # the training version; workers are fresh processes, so no restart

In [ ]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    tok = None
login(token=tok) if tok else login()

In [ ]:
%%writefile augment.py
# -*- coding: utf-8 -*-
"""Augmentation families x strength levels for the augmentation / training-free-metrics study.

ONE source of truth: the training notebooks (online augmentation) and the metric computation (K augmented draws of the training
set) both call `augment()` from this file, so the metrics describe exactly the distribution the recognizer was trained on.

Input / output: a grayscale uint8 line image (H x W, white background, dark ink) ALREADY resized to the model line height
(64 px), so every magnitude below is in pixels of a 64 px high line. Level 0 = identity. For level L >= 1 every image is
augmented (p = 1); each magnitude is drawn uniformly from [0, MAX[L]] (with a random sign where it has one), so a higher
level widens the distribution rather than shifting it.

Families
  geom     rotation U(-a, a) deg + horizontal shear U(-s, s); the canvas is enlarged so no ink is cut, then resized back to 64 px
           high (a rotated long line gets taller, so its letters shrink: part of what rotating a cropped line does)
  elastic  smooth random displacement field (Gaussian-blurred noise, sigma 4 px), RMS displacement U(0, A) px per axis
  stroke   ink thickening with an elliptical kernel of size r+1, r ~ U{-1..k}; r = -1 thins by one px (more thinning erases
           HHD's ~3 px strokes), r = 0 leaves the line unchanged
  photo    Gaussian blur sigma U(0, b) + ink-contrast loss (ink lightened by up to c) + Gaussian pixel noise std U(0, n)
"""
import math

import cv2
import numpy as np

LEVELS = (0, 1, 2, 3, 4)
FAMILIES = ("geom", "elastic", "stroke", "photo")
MAX = {
    "geom":    {"rot": (0, 0.75, 1.5, 2.25, 3.0), "shear": (0, 0.10, 0.20, 0.35, 0.50)},
    "elastic": {"amp": (0, 0.75, 1.5, 2.25, 3.0), "sigma": 4.0},
    "stroke":  {"k": (0, 1, 2, 3, 4)},
    "photo":   {"blur": (0, 0.5, 1.0, 1.5, 2.0), "contrast": (0, 0.15, 0.30, 0.45, 0.60), "noise": (0, 0.03, 0.06, 0.10, 0.15)},
}


def condition_name(family, level):
    return "none" if level == 0 or family in (None, "none") else f"{family}{level}"


def parse_condition(name):
    """'none' -> (None, 0); 'geom3' -> ('geom', 3)"""
    if name == "none": return None, 0
    return name[:-1], int(name[-1])


def _geom(g, L, rng):
    a = math.radians(rng.uniform(-1, 1) * MAX["geom"]["rot"][L]); s = rng.uniform(-1, 1) * MAX["geom"]["shear"][L]
    h0, w0 = g.shape
    px = int(math.ceil(abs(s) * h0 + abs(math.sin(a)) * h0)) + 2; py = int(math.ceil(abs(math.sin(a)) * w0 / 2)) + 1
    g = cv2.copyMakeBorder(g, py, py, px, px, cv2.BORDER_CONSTANT, value=255); h, w = g.shape
    cx, cy = w / 2, h / 2
    R = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]]) @ np.array([[1, s], [0, 1]])
    M = np.hstack([R, (np.array([cx, cy]) - R @ np.array([cx, cy]))[:, None]])
    g = cv2.warpAffine(g, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=255)
    rows = np.where((g < 200).any(1))[0]                       # crop the vertical padding back to the ink (+ the original margin)
    if len(rows): g = g[max(0, rows[0] - 2):rows[-1] + 3]
    return cv2.resize(g, (max(1, round(g.shape[1] * h0 / g.shape[0])), h0), interpolation=cv2.INTER_AREA) if g.shape[0] != h0 else g


def _elastic(g, L, rng):
    amp = rng.uniform(0, MAX["elastic"]["amp"][L]); sig = MAX["elastic"]["sigma"]
    h, w = g.shape
    f = [cv2.GaussianBlur(rng.standard_normal((h, w)).astype(np.float32), (0, 0), sig) for _ in range(2)]
    dx, dy = (f[i] * (amp / max(1e-6, float(f[i].std()))) for i in (0, 1))
    xs, ys = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    return cv2.remap(g, xs + dx, ys + dy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=255)


def _stroke(g, L, rng):
    k = MAX["stroke"]["k"][L]; r = int(rng.integers(-1, k + 1))
    if r == 0: return g
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (abs(r) + 1, abs(r) + 1))
    return cv2.erode(g, ker) if r > 0 else cv2.dilate(g, ker)   # white background: erode = bolder ink, dilate = thinner ink


def _photo(g, L, rng):
    P = MAX["photo"]; x = g.astype(np.float32)
    b = rng.uniform(0, P["blur"][L])
    if b > 0.05: x = cv2.GaussianBlur(x, (0, 0), b)
    c = rng.uniform(0, P["contrast"][L]); x = 255 - (255 - x) * (1 - c)
    n = rng.uniform(0, P["noise"][L])
    if n > 0: x = x + rng.standard_normal(x.shape).astype(np.float32) * (255 * n)
    return np.clip(x, 0, 255).astype(np.uint8)


_FN = {"geom": _geom, "elastic": _elastic, "stroke": _stroke, "photo": _photo}


def augment(gray, family, level, rng):
    """gray: uint8 H x W (64 px high, white background). Returns a uint8 image of the same height (geom may widen it)."""
    if level == 0 or family in (None, "none"): return gray
    return _FN[family](gray, level, rng)


def to_line(image, height=64, max_width=None):
    """PIL image -> grayscale uint8 array, `height` px high (aspect kept; optionally squeezed to max_width)."""
    g = np.array(image.convert("L")); h, w = g.shape
    nw = max(1, round(w * height / h))
    if max_width and nw > max_width: nw = max_width
    return cv2.resize(g, (nw, height), interpolation=cv2.INTER_AREA if h > height else cv2.INTER_CUBIC)


In [ ]:
%%writefile metrics_hmt.py
# -*- coding: utf-8 -*-
"""Training-free metrics of every wave-1 augmentation condition, against the dev reference.

For each condition (none + 4 families x 4 levels) and draw k (K draws; `none` has no randomness -> 1 draw) the whole
training set (Option-B train minus the 20% dev writers, exactly the lines the recognizers were trained on) is augmented
once with augment.py and embedded; every metric is computed against the un-augmented dev lines (the reference).

Same protocol as the paper scorer (scorer-ocrs-dataset / paper_v3/analysis/score_all.py), code ported unchanged:
  granularities  patch (Otsu, 64 px, <=3 square windows/line), word (gap segmentation, gap calibrated on 300 train lines),
                 line (ink crop, pad to square), vit_token (TrOCR-base encoder tokens on the 384 tiled container)
  encoder        DINOv2-small (vit_small_patch14_dinov2.lvd142m, 224) for patch/word/line; TrOCR-base-handwritten for vit_token
  metrics        coverage, density, precision (k=5), Vendi + mean pairwise distance (2000-vector subsample),
                 Vendi SD over 10 subsamples, proxy A-distance (5-fold logistic regression), KID x1000 (10 x 1000)
  caps           every candidate granularity capped at 6000 vectors (seed 42); vit_token draws 6000 tokens from all lines

One JSON per (condition, draw) is pushed to HF dataset OUT_REPO/per/; a run skips files already there (resumable).
Usage: python metrics_hmt.py --shard i --nshards n [--draws 3] [--smoke]
"""
import argparse, io, json, os, sys, time

import numpy as np
import torch
from PIL import Image, ImageOps

import augment as A

OUT_REPO = "cyttic/hmt-metrics"
SEED, M_CAP, TOKEN_CAP, CHUNK_LINES, N_CALIB, ROW_H = 42, 6000, 6000, 100, 300, 64
DINO = "vit_small_patch14_dinov2.lvd142m"
TROCR = "microsoft/trocr-base-handwritten"
DEV = "cuda:0"


# ---------------------------------------------------------------- preprocessing (compare_datasets / common.py)
def otsu_binarize(img):
    a = np.asarray(img)
    hist = np.bincount(a.ravel(), minlength=256).astype(float)
    w = hist.cumsum(); m = (hist * np.arange(256)).cumsum()
    w1 = w[-1] - w; mu0 = np.where(w > 0, m / np.maximum(w, 1e-9), 0)
    mu1 = np.where(w1 > 0, (m[-1] - m) / np.maximum(w1, 1e-9), 0)
    t = int(np.argmax(w * w1 * (mu0 - mu1) ** 2))
    return Image.fromarray(np.where(a <= t, 0, 255).astype(np.uint8))


def patches(img, side, max_p=3, min_ink=0.01):
    a = np.asarray(img); ys, xs = np.where(a < 200)
    if len(ys) < 10: return []
    m = 4
    img = img.crop((max(0, xs.min() - m), max(0, ys.min() - m), min(img.width, xs.max() + 1 + m), min(img.height, ys.max() + 1 + m)))
    w, h = img.size; nw = max(side, int(w * side / h)); img = img.resize((nw, side), Image.LANCZOS)
    xs = [0] if nw <= side else list(np.linspace(0, nw - side, min(max_p, 1 + nw // side)).astype(int))
    res = []
    for x in xs:
        p = img.crop((x, 0, x + side, side))
        if (np.asarray(p) < 200).mean() >= min_ink: res.append(p.convert("RGB"))
    return res


def resize_to_h(img, h=64):
    w0, h0 = img.size
    return img if h0 == h else img.resize((max(1, int(w0 * h / h0)), h), Image.LANCZOS)


def ink_crop(img, margin=4):
    a = np.asarray(img); ys, xs = np.where(a < 200)
    if len(ys) < 10: return None
    return img.crop((max(0, xs.min() - margin), max(0, ys.min() - margin), min(img.width, xs.max() + 1 + margin), min(img.height, ys.max() + 1 + margin)))


def pad_to_square_resize(img, target=224, bg=255):
    w, h = img.size; s = target / max(w, h); nw, nh = max(1, int(w * s)), max(1, int(h * s))
    c = Image.new("L", (target, target), bg); c.paste(img.resize((nw, nh), Image.LANCZOS), ((target - nw) // 2, (target - nh) // 2))
    return c


def resize_h64_pad(img, target_h=64, canvas=224, bg=255):
    w, h = img.size; nw = max(1, int(w * target_h / h)); r = img.resize((nw, target_h), Image.LANCZOS)
    out = Image.new("L", (canvas, canvas), bg); x0 = max(0, (canvas - nw) // 2)
    if nw > canvas:
        cx0 = (nw - canvas) // 2; r = r.crop((cx0, 0, cx0 + canvas, target_h)); x0 = 0
    out.paste(r, (x0, (canvas - target_h) // 2))
    return out


def segment_words(img, gap_thresh):
    b = otsu_binarize(resize_to_h(img, 64)); ink = (np.asarray(b) < 128).any(axis=0)
    segs, in_word, gap, start, W = [], False, 0, 0, len(ink)
    for x in range(W):
        if ink[x]:
            if not in_word: start, in_word = x, True
            gap = 0
        elif in_word:
            gap += 1
            if gap > gap_thresh: segs.append((start, x - gap + 1)); in_word, gap = False, 0
    if in_word: segs.append((start, W - gap if gap else W))
    return [(s, e) for s, e in segs if e > s], b


def crop_word(b, x0, x1, margin=3):
    a = np.asarray(b); ys = np.where((a[:, max(0, x0 - margin):min(a.shape[1], x1 + margin)] < 128).any(axis=1))[0]
    y0, y1 = (0, a.shape[0]) if len(ys) == 0 else (max(0, ys.min() - margin), min(a.shape[0], ys.max() + 1 + margin))
    return b.crop((max(0, x0 - margin), y0, min(a.shape[1], x1 + margin), y1))


def calibrate_gap(lines):
    med_words = np.median([len(t.split()) for _, t in lines]); best = None
    for g in range(1, 41):
        d = abs(np.median([len(segment_words(im, g)[0]) for im, _ in lines]) - med_words)
        if best is None or d < best[0]: best = (d, g)
    return best[1]


# ---------------------------------------------------------------- embedders
class Dino:
    def __init__(self):
        import timm
        from timm.data import resolve_data_config, create_transform
        self.model = timm.create_model(DINO, pretrained=True, num_classes=0, img_size=224).to(DEV).eval()
        cfg = resolve_data_config({}, model=self.model); cfg["input_size"] = (3, 224, 224)
        self.tf, self.side = create_transform(**cfg), 224

    @torch.no_grad()
    def __call__(self, imgs, bs=64):
        feats = []
        for s in range(0, len(imgs), bs):
            x = torch.stack([self.tf(im.convert("RGB")) for im in imgs[s:s + bs]]).to(DEV)
            feats.append(self.model(x).float().cpu().numpy())
        E = np.concatenate(feats) if feats else np.zeros((0, 384), np.float32)
        return E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)


def tile384(img):
    """HebrewBlockProcessor with AUTOCROP off: mirror (RTL->LTR), 64 px high, tiled row by row into 384x384, [-1, 1]."""
    img = ImageOps.mirror(img.convert("RGB")); w, h = img.size; nw = max(1, round(w * 64 / h))
    a = np.array(img.resize((nw, 64), Image.LANCZOS)); c = Image.new("RGB", (384, 384), (255, 255, 255))
    sx = dx = dy = 0
    while sx < nw and dy < 384:
        cw = min(nw - sx, 384 - dx); c.paste(Image.fromarray(a[:, sx:sx + cw]), (dx, dy)); sx += cw; dx += cw
        if dx >= 384: dx, dy = 0, dy + 64
    return (torch.tensor(np.array(c), dtype=torch.float32).permute(2, 0, 1) / 255.0 - 0.5) / 0.5


class TrocrTokens:
    def __init__(self):
        from transformers import VisionEncoderDecoderModel
        self.enc = VisionEncoderDecoderModel.from_pretrained(TROCR).encoder.to(DEV).eval()

    @torch.no_grad()
    def __call__(self, imgs, bs=16):
        feats = []
        for s in range(0, len(imgs), bs):
            tok = self.enc(torch.stack([tile384(im) for im in imgs[s:s + bs]]).to(DEV)).last_hidden_state[:, 1:, :]
            feats.append(tok.reshape(-1, tok.shape[-1]).float().cpu().numpy())
        E = np.concatenate(feats) if feats else np.zeros((0, 768), np.float32)
        return E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)


def token_sample(vit, imgs, seed=SEED):
    rng = np.random.default_rng(seed); nch = -(-len(imgs) // CHUNK_LINES); per = -(-TOKEN_CAP // nch); out = []
    for c in range(nch):
        e = vit(imgs[c * CHUNK_LINES:(c + 1) * CHUNK_LINES]); out.append(e[rng.choice(len(e), min(per, len(e)), replace=False)])
    X = np.concatenate(out)
    return X[rng.choice(len(X), TOKEN_CAP, replace=False)] if len(X) > TOKEN_CAP else X


def cap(E, m=M_CAP, seed=SEED):
    return E if len(E) <= m else E[np.random.default_rng(seed).choice(len(E), m, replace=False)]


def embed_all(lines, dino, vit, gap):
    """lines: list of (PIL L image, text). Returns {granularity: vectors}, word retention."""
    imgs = [im for im, _ in lines]
    P = [p for im in imgs for p in patches(otsu_binarize(resize_to_h(im, 64)), dino.side)]
    W, kept = [], 0
    for im, t in lines:
        segs, b = segment_words(im, gap)
        if len(segs) != len(t.split()): continue
        crops = [crop_word(b, x0, x1) for x0, x1 in segs]
        if any(c.width < 2 or c.height < 2 for c in crops): continue
        kept += 1; W += [resize_h64_pad(c) for c in crops]
    L = [pad_to_square_resize(c) for c in (ink_crop(otsu_binarize(im)) for im in imgs) if c is not None]
    X = {"patch": cap(dino(P)), "word": cap(dino(W)), "line": cap(dino(L)), "vit_token": token_sample(vit, imgs)}
    return X, kept / len(lines)


# ---------------------------------------------------------------- metrics (compare_datasets.py, unchanged)
def kid(X, Y, subset=1000, reps=10, seed=0):
    d = X.shape[1]; rng = np.random.default_rng(seed); vals = []
    k = lambda a, b: (a @ b.T / d + 1.0) ** 3
    for _ in range(reps):
        x = X[rng.choice(len(X), min(subset, len(X)), replace=False)]; y = Y[rng.choice(len(Y), min(subset, len(Y)), replace=False)]
        m, n = len(x), len(y); kxx, kyy, kxy = k(x, x), k(y, y), k(x, y); np.fill_diagonal(kxx, 0); np.fill_diagonal(kyy, 0)
        vals.append(kxx.sum() / (m * (m - 1)) + kyy.sum() / (n * (n - 1)) - 2 * kxy.mean())
    return float(np.mean(vals)) * 1000, float(np.std(vals)) * 1000


def coverage_density(X, R, k=5):
    from sklearn.neighbors import NearestNeighbors
    from sklearn.metrics import pairwise_distances
    radii = NearestNeighbors(n_neighbors=k + 1).fit(R).kneighbors(R)[0][:, -1]
    d = pairwise_distances(R, X); within = d < radii[:, None]
    return float((d.min(axis=1) < radii).mean()), float(within.sum(axis=0).mean() / k), float(within.any(axis=0).mean())


def vendi(X, cap_=2000, seed=0):
    x = X[np.random.default_rng(seed).choice(len(X), min(cap_, len(X)), replace=False)]
    ev = np.clip(np.linalg.eigvalsh(x @ x.T / len(x)), 1e-12, None); ev /= ev.sum()
    return float(np.exp(-(ev * np.log(ev)).sum()))


def mean_pairwise(X, cap_=2000, seed=0):
    x = X[np.random.default_rng(seed).choice(len(X), min(cap_, len(X)), replace=False)]
    S = x @ x.T; n = len(x)
    return float(1 - (S.sum() - np.trace(S)) / (n * (n - 1)))


def a_distance(X, R, seed=0):
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    n = min(len(X), len(R)); rng = np.random.default_rng(seed)
    Z = np.concatenate([X[rng.choice(len(X), n, replace=False)], R[rng.choice(len(R), n, replace=False)]])
    acc = cross_val_score(LogisticRegression(max_iter=2000), Z, np.r_[np.zeros(n), np.ones(n)], cv=5).mean()
    return float(2 * (2 * acc - 1))


def score(X, R):
    cov, dens, prec = coverage_density(X, R); km, ks = kid(X, R)
    return dict(Cover=cov, Dens=dens, Prec=prec, Vendi=vendi(X), Vendi_sd=float(np.std([vendi(X, seed=s) for s in range(10)])),
                PairD=mean_pairwise(X), A_dist=a_distance(X, R), KID=km, KID_sd=ks, n=int(len(X)))


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--shard", type=int, default=0); ap.add_argument("--nshards", type=int, default=1)
    ap.add_argument("--draws", type=int, default=3); ap.add_argument("--smoke", action="store_true")
    a = ap.parse_args()
    from datasets import load_dataset
    from huggingface_hub import HfApi, create_repo
    api = HfApi(); create_repo(OUT_REPO, repo_type="dataset", exist_ok=True)
    done = {f for f in api.list_repo_files(OUT_REPO, repo_type="dataset")}

    hhd = load_dataset("cyttic/hhd-optionb-split")["train"]
    tw = list(hhd["writer"]); wr = sorted(set(tw)); __import__("random").Random(42).shuffle(wr); dev_w = set(wr[:int(len(wr) * 0.20)])
    to = lambda ex: (Image.fromarray(A.to_line(ex["image"], ROW_H)), ex["text"])
    train = [to(hhd[i]) for i, w in enumerate(tw) if w not in dev_w]          # (64 px L image, text): what the models trained on
    dev = [to(hhd[i]) for i, w in enumerate(tw) if w in dev_w]
    if a.smoke: train, dev = train[:120], dev[:60]
    print(f"train {len(train)} lines | dev {len(dev)} lines (reference)", flush=True)

    conds = ["none"] + [f"{f}{l}" for f in A.FAMILIES for l in A.LEVELS[1:]]
    jobs = [(c, k) for c in conds for k in range(1 if c == "none" else a.draws)]
    mine = [j for i, j in enumerate(jobs) if i % a.nshards == a.shard and f"per/{j[0]}_k{j[1]}.json" not in done]
    print(f"shard {a.shard}/{a.nshards}: {len(mine)} of {len(jobs)} jobs to do", flush=True)
    if not mine: return

    dino, vit = Dino(), TrocrTokens()
    gap = calibrate_gap(train[:N_CALIB])
    t0 = time.time(); R, ret_dev = embed_all(dev, dino, vit, gap)
    print(f"gap {gap} | reference sizes { {g: len(v) for g, v in R.items()} } | {time.time() - t0:.0f}s", flush=True)

    for c, k in mine:
        t0 = time.time(); fam, lvl = A.parse_condition(c); rng = np.random.default_rng([SEED, 1000 + k, conds.index(c)])
        cand = [(Image.fromarray(A.augment(np.asarray(im), fam, lvl, rng)), t) for im, t in train]
        X, ret = embed_all(cand, dino, vit, gap)
        res = dict(condition=c, family=fam or "none", level=lvl, draw=k, n_lines=len(cand), word_retention=ret,
                   dev_word_retention=ret_dev, gap_thresh=int(gap), protocol=dict(encoder=DINO, vit_token=TROCR, M_CAP=M_CAP,
                   TOKEN_CAP=TOKEN_CAP, seed=SEED, reference="dev writers (20% of Option-B train, seed 42), un-augmented"),
                   **{g: score(X[g], R[g]) for g in ("patch", "word", "line", "vit_token")})
        os.makedirs("per", exist_ok=True); json.dump(res, open(f"per/{c}_k{k}.json", "w"), indent=1)   # local copy too
        for attempt in range(5 if not a.smoke else 0):                    # two shards commit to one repo + flaky network: retry
            try:
                api.upload_file(path_or_fileobj=f"per/{c}_k{k}.json", path_in_repo=f"per/{c}_k{k}.json",
                                repo_id=OUT_REPO, repo_type="dataset", commit_message=f"{c} draw {k}"); break
            except Exception as e:
                print(f"upload retry {attempt + 1}: {e}", flush=True); time.sleep(30)
        print(f"[{c} k{k}] {time.time() - t0:.0f}s  Vendi p/w/l/t " + " ".join(f"{res[g]['Vendi']:.4f}" for g in ("patch", "word", "line", "vit_token"))
              + f"  KID line {res['line']['KID']:.4f}  ret {ret:.4f}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile lofo.py
# -*- coding: utf-8 -*-
"""Leave-one-family-out test: can training-free image metrics predict the test CER of an augmentation they were not fitted on?

CER  = final.test_greedy.cer of every HF run cyttic/hmt-<model>-<cond>-s<seed> (fetched -> results/cer.json).
X    = results/metrics.csv (metrics_hmt.py on Kaggle, cyttic/hmt-metrics; image-only, so the same for every model), mean over draws,
       per granularity (patch / word / line / vit_token); combos: Vendi alone, Vendi + one other metric, two triples, Vendi x A_dist.
LOFO = for each family (geom / elastic / stroke / photo): OLS on z-scored X fitted on the other 3 families + 'none', predict its 4 levels.
Scores on the 16 augmented conditions: MAE (CER points) and Spearman rho (prediction vs CER). Baselines: mean of the other families;
strength level only. Permutation test (300 shuffles of CER over the 16 conditions): ratio best-combo MAE / baseline MAE vs its null
(the best combo is re-picked on every shuffle, so the search over 4 x 10 combos is accounted for).
Targets: trocrctc s42, trocrsmall s42, trocrscratch s42 (TrOCR-small architecture, random init), htrvt s42, htrvt s43, htrvt mean of s42+s43; plus seed agreement of htrvt (s42 vs s43).

    /mnt/ssd2/cyttic/ml_env/bin/python lofo.py
-> results/cer.json, results/lofo.json
"""
import json, os, time

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

HERE = os.path.dirname(os.path.abspath(__file__)); RES = os.path.join(HERE, "results")
FAMS = ["geom", "elastic", "stroke", "photo"]
CONDS = ["none"] + [f"{f}{l}" for f in FAMS for l in range(1, 5)]
fam = np.array(["none"] + [f for f in FAMS for _ in range(4)]); lvl = np.array([0] + [l for _ in FAMS for l in range(1, 5)], float)
aug = fam != "none"
M = ["Vendi", "PairD", "KID", "Cover", "Dens", "Prec", "A_dist"]
COMBOS = [("Vendi",)] + [("Vendi", k) for k in M[1:]] + [("Vendi", "KID", "Prec"), ("Vendi", "A_dist", "Cover"), ("Vendi_x_A",)]


def fetch_cer():
    from huggingface_hub import HfApi, hf_hub_download
    out = {}
    for m in sorted(x.id for x in HfApi().list_models(author="cyttic", search="hmt-")):
        if m.endswith("hmt-metrics"): continue
        for k in range(3):
            try: out[m.split("/")[1][4:]] = json.load(open(hf_hub_download(m, "results.json")))["final"]["test_greedy"]["cer"]; break
            except Exception: time.sleep(5)          # flaky router DNS
    json.dump(out, open(os.path.join(RES, "cer.json"), "w"), indent=1)
    return out


def lofo(X, y):
    pred = np.full(len(y), np.nan)
    for f in FAMS:
        te = fam == f; tr = ~te
        mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-12; A = np.c_[np.ones(tr.sum()), (X[tr] - mu) / sd]
        w = np.linalg.lstsq(A, y[tr], rcond=None)[0]; pred[te] = np.c_[np.ones(te.sum()), (X[te] - mu) / sd] @ w
    return pred


mae = lambda p, y: float(np.abs(p[aug] - y[aug]).mean())


def main():
    cer = fetch_cer()
    df = pd.read_csv(os.path.join(RES, "metrics.csv")); T = {}
    for g in ["patch", "word", "line", "vit_token"]:
        t = df[df.granularity == g].groupby("condition")[M].mean().loc[CONDS]; t["Vendi_x_A"] = t.Vendi * t.A_dist; T[g] = t
    Y = {"trocrctc_s42": [cer[f"trocrctc-{c}-s42"] for c in CONDS], "trocrsmall_s42": [cer[f"trocrsmall-{c}-s42"] for c in CONDS],
         "trocrscratch_s42": [cer[f"trocrscratch-{c}-s42"] for c in CONDS],
         "htrvt_s42": [cer[f"htrvt-{c}-s42"] for c in CONDS], "htrvt_s43": [cer[f"htrvt-{c}-s43"] for c in CONDS]}
    if all(f"trocrscratchreg-{c}-s42" in cer for c in CONDS):          # ladder A, once all 17 runs are on HF
        Y["trocrscratchreg_s42"] = [cer[f"trocrscratchreg-{c}-s42"] for c in CONDS]
    Y["htrvt_mean"] = list((np.array(Y["htrvt_s42"]) + np.array(Y["htrvt_s43"])) / 2)
    Y = {k: 100 * np.array(v) for k, v in Y.items()}
    a, b = Y["htrvt_s42"], Y["htrvt_s43"]
    out = {"htrvt_seed_agreement": {"spearman_16aug": float(spearmanr(a[aug], b[aug])[0]), "mean_abs_diff": float(np.abs(a - b).mean())}}
    print(f"HTR-VT s42 vs s43 on the 16 augmented conditions: rho {out['htrvt_seed_agreement']['spearman_16aug']:.4f}, "
          f"mean |diff| {out['htrvt_seed_agreement']['mean_abs_diff']:.2f} CER points")

    def best_ratio(y):
        b0 = mae(lofo(np.zeros((17, 0)), y), y)
        return min(mae(lofo(T[g][list(c)].values, y), y) for g in T for c in COMBOS) / b0

    rng = np.random.default_rng(0)
    for name, y in Y.items():
        base = lofo(np.zeros((17, 0)), y); lv = lofo(lvl[:, None], y)
        rows = [{"gran": g, "metrics": "+".join(c), "MAE": mae(p, y), "rho": float(spearmanr(p[aug], y[aug])[0])}
                for g in T for c in COMBOS for p in [lofo(T[g][list(c)].values, y)]]
        rows.sort(key=lambda r: r["MAE"]); best = rows[0]
        obs = best_ratio(y); null = []
        for _ in range(300):
            yp = y.copy(); yp[aug] = rng.permutation(y[aug]); null.append(best_ratio(yp))
        null = np.array(null)
        bp = lofo(T[best["gran"]][best["metrics"].split("+")].values, y)
        out[name] = {"baseline_mean_MAE": mae(base, y), "baseline_level_MAE": mae(lv, y), "baseline_level_rho": float(spearmanr(lv[aug], y[aug])[0]),
                     "best": best, "top5": rows[:5], "best_over_baseline": obs, "perm_median": float(np.median(null)), "perm_p": float((null <= obs).mean()),
                     "best_MAE_per_family": {f: float(np.abs(bp[fam == f] - y[fam == f]).mean()) for f in FAMS}}
        o = out[name]
        print(f"\n== {name}: baseline mean MAE {o['baseline_mean_MAE']:.2f} | level MAE {o['baseline_level_MAE']:.2f} (rho {o['baseline_level_rho']:.4f})")
        for r in rows[:5]: print(f"   {r['gran']:9s} {r['metrics']:20s} MAE {r['MAE']:.2f} rho {r['rho']:.4f}")
        print(f"   best/baseline {obs:.4f} | permuted median {o['perm_median']:.4f} | p {o['perm_p']:.4f} | per family "
              + ", ".join(f"{f} {v:.2f}" for f, v in o["best_MAE_per_family"].items()))
    json.dump(out, open(os.path.join(RES, "lofo.json"), "w"), indent=1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile model_aware.py
# -*- coding: utf-8 -*-
"""Model-aware predictors of augmentation benefit (H2: why do the image-only metrics predict HTR-VT's CER but not TrOCR's?).

Each recognizer trained WITHOUT augmentation (cyttic/hmt-<arch>-none-s<seed>) is the probe; nothing is trained here.
  affinity  -- its greedy CER on the dev lines augmented with each condition (3 draws; Gontijo-Lopes et al., ICLR 2021), minus the
               clean-dev CER: how far the augmentation moves the data away from what this model already reads
  own       -- the training-free metrics of metrics_hmt.py (score(): Cover/Dens/Prec/Vendi/PairD/A_dist/KID), computed in the model's
               OWN feature space = the input of its CTC classifier. frame: 6000 valid frames sampled over all lines; line: the mean of
               each line's valid frames. Candidates = the training set augmented with exactly metrics_hmt.py's draws (same rng);
               reference = the clean dev lines. Vectors are L2-normalised; above 1024 dims (TrOCR frames = 4 patch rows x hidden)
               they are first PCA-reduced to 768 dims, fitted once on the model's dev reference.
Model and preprocessing code are the training notebooks' own (model_src.json, written by make_model_aware_notebook.py).

    python model_aware.py --shard i --nshards n     one worker per GPU; each finished job -> HF cyttic/hmt-metrics model_aware/
    python model_aware.py --analyze                 merge + leave-one-family-out test -> model_aware/summary.json
"""
import argparse, json, math, os, time

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

import augment as A

OUT_REPO, PREFIX, SEED, CAP, ROW_H = "cyttic/hmt-metrics", "model_aware", 42, 6000, 64
DEV = "cuda:0"
PROBES = {"htrvt-s42": ("htrvt", 42), "htrvt-s43": ("htrvt", 43), "trocrctc-s42": ("trocrctc", 42),
          "trocrsmall-s42": ("trocrsmall", 42), "trocrscratch-s42": ("trocrscratch", 42)}
CONDS = ["none"] + [f"{f}{l}" for f in A.FAMILIES for l in A.LEVELS[1:]]
DRAWS = 3


def load_split():
    from datasets import load_dataset
    from huggingface_hub import hf_hub_download
    chars = json.load(open(hf_hub_download("cyttic/exp35-trocr-large-en-ctc-seqhead-allreal-synpunct", "charset.json")))["chars"]
    hhd = load_dataset("cyttic/hhd-optionb-split")["train"]
    tw = list(hhd["writer"]); wr = sorted(set(tw)); __import__("random").Random(42).shuffle(wr); dev_w = set(wr[:int(len(wr) * 0.20)])
    train = [A.to_line(hhd[i]["image"], ROW_H) for i, w in enumerate(tw) if w not in dev_w]   # same order as metrics_hmt.py
    dev = [(A.to_line(hhd[i]["image"], ROW_H), hhd[i]["text"]) for i, w in enumerate(tw) if w in dev_w]
    return chars, train, dev


def build(arch, seed, n_classes):
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    S = json.load(open("model_src.json"))[arch]
    ns = {"np": np, "torch": torch, "nn": nn, "F": F, "math": math, "os": os, "A": A, "SEED": seed, "N_CLASSES": n_classes, "device": DEV}
    for k in ("cfg", "fmt", "model"): exec(S[k], ns)
    ns["model"].load_state_dict(load_file(hf_hub_download(f"cyttic/hmt-{arch}-none-s{seed}", "model.safetensors")))
    ns["model"].eval()
    return ns


def greedy(lp, n, chars):
    out, prev = [], 0
    for k in lp[:n].argmax(-1).tolist():
        if k != prev and k != 0: out.append(chars[k])
        prev = k
    return "".join(out)


@torch.no_grad()
def run(ns, grays, chars, rng, bs=32):
    """-> greedy hypotheses, sampled frames (<= CAP over all lines), per-line mean frames. Frames = the CTC classifier's input."""
    model = ns["model"]; head = model.classifier if hasattr(model, "classifier") else model.head; sub = ns.get("SUB", 1)
    per = -(-CAP // len(grays)) + 1; cap = {}; hyps, frames, means = [], [], []
    hk = head.register_forward_hook(lambda m, i, o: cap.__setitem__("x", i[0]))
    for s in range(0, len(grays), bs):
        items = [ns["fmt"](g) for g in grays[s:s + bs]]
        px, n = ns["collate"]([(x, k, [], "") for x, k in items])[:2]
        with torch.autocast("cuda", dtype=torch.float16):
            lp = model(px.to(DEV), n.to(DEV))
        lp = lp.float().cpu().numpy(); X = cap["x"].float().cpu().numpy()
        for i in range(len(items)):
            k = int(n[i]); hyps.append(greedy(lp[i], k, chars)); f = X[i, :max(1, k // sub)]
            frames.append(f[rng.choice(len(f), min(per, len(f)), replace=False)]); means.append(f.mean(0))
    hk.remove()
    F_ = np.concatenate(frames)
    return hyps, (F_[rng.choice(len(F_), CAP, replace=False)] if len(F_) > CAP else F_), np.stack(means)


def cer(refs, hyps):
    import jiwer
    k = [i for i, r in enumerate(refs) if r.strip()]
    return float(jiwer.cer([refs[i] for i in k], [hyps[i] for i in k]))


def projector(R):
    """Fitted on the dev reference: PCA to 768 dims above 1024 dims, then L2 normalisation."""
    if R.shape[1] > 1024:
        mu = R.mean(0); V = np.linalg.svd(R - mu, full_matrices=False)[2][:768]
        f = lambda X: (X - mu) @ V.T
    else:
        f = lambda X: X
    return lambda X: (lambda Z: Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9))(f(X)).astype(np.float32)


def work(shard, nshards, smoke):
    from huggingface_hub import HfApi, create_repo
    from metrics_hmt import score
    api = HfApi(); create_repo(OUT_REPO, repo_type="dataset", exist_ok=True)
    done = set(api.list_repo_files(OUT_REPO, repo_type="dataset"))
    jobs = [(p, c, k) for p in PROBES for c in CONDS for k in range(1 if c == "none" else DRAWS)]
    mine = [j for i, j in enumerate(jobs) if i % nshards == shard and f"{PREFIX}/{j[0]}/{j[1]}_k{j[2]}.json" not in done]
    print(f"shard {shard}/{nshards}: {len(mine)} of {len(jobs)} jobs to do", flush=True)
    if not mine: return
    chars, train, dev = load_split()
    if smoke: train, dev = train[:96], dev[:48]
    refs, dev_g = [t for _, t in dev], [g for g, _ in dev]
    print(f"train {len(train)} | dev {len(dev)}", flush=True)
    cur = None
    for p, c, k in mine:
        t0 = time.time()
        if cur != p:                                             # jobs are grouped by probe: load it once, embed the clean dev once
            arch, seed = PROBES[p]; ns = build(arch, seed, len(chars)); cur = p
            h0, Rf, Rl = run(ns, dev_g, chars, np.random.default_rng([SEED, 7]))
            clean = cer(refs, h0); proj = {"frame": projector(Rf), "line": projector(Rl)}
            R = {"frame": proj["frame"](Rf), "line": proj["line"](Rl)}
            print(f"[{p}] loaded | clean dev CER {100 * clean:.2f} | frame dim {Rf.shape[1]}", flush=True)
        fam, lvl = A.parse_condition(c); ci = CONDS.index(c)
        rng = np.random.default_rng([SEED, 1000 + k, ci])        # = metrics_hmt.py: the same augmented training set
        cand = [A.augment(g, fam, lvl, rng) for g in train]
        _, Xf, Xl = run(ns, cand, chars, np.random.default_rng([SEED, 8, ci, k]))
        rng = np.random.default_rng([SEED, 2000 + k, ci])        # affinity: the dev lines, augmented
        hyp, _, _ = run(ns, [A.augment(g, fam, lvl, rng) for g in dev_g], chars, np.random.default_rng([SEED, 9]))
        res = dict(probe=p, condition=c, family=fam or "none", level=lvl, draw=k, clean_dev_cer=clean, aug_dev_cer=cer(refs, hyp),
                   frame=score(proj["frame"](Xf), R["frame"]), line=score(proj["line"](Xl), R["line"]))
        path = f"{PREFIX}/{p}/{c}_k{k}.json"; os.makedirs(os.path.dirname(path), exist_ok=True); json.dump(res, open(path, "w"), indent=1)
        for attempt in range(0 if smoke else 5):                # two workers commit to one repo + flaky network: retry
            try: api.upload_file(path_or_fileobj=path, path_in_repo=path, repo_id=OUT_REPO, repo_type="dataset", commit_message=f"{p} {c} k{k}"); break
            except Exception as e: print(f"upload retry {attempt + 1}: {e}", flush=True); time.sleep(30)
        print(f"[{p} {c} k{k}] {time.time() - t0:.0f}s | dev CER {100 * res['aug_dev_cer']:.2f} | Vendi frame/line "
              f"{res['frame']['Vendi']:.4f}/{res['line']['Vendi']:.4f} | KID line {res['line']['KID']:.4f}", flush=True)


def analyze():
    """Leave-one-family-out (lofo.py's protocol and permutation test) with three predictor sets per probe:
    affinity alone; own-feature metrics (frame/line x lofo.COMBOS); affinity + the fixed image-only combo (word Vendi+KID+Prec)."""
    import glob, pandas as pd
    from huggingface_hub import HfApi, hf_hub_download, snapshot_download
    from scipy.stats import spearmanr
    import lofo as L
    os.makedirs(L.RES, exist_ok=True)
    d = snapshot_download(OUT_REPO, repo_type="dataset", allow_patterns=[f"{PREFIX}/*/*.json", "metrics.csv"])
    rows = [json.load(open(f)) for f in glob.glob(f"{d}/{PREFIX}/*/*.json")]
    print(len(rows), "of", len(PROBES) * (1 + 16 * DRAWS), "jobs present")
    cer = L.fetch_cer()
    img = pd.read_csv(f"{d}/metrics.csv"); img = img[img.granularity == "word"].groupby("condition")[["Vendi", "KID", "Prec"]].mean().loc[L.CONDS]
    out, rng = {}, np.random.default_rng(0)
    for p, (arch, seed) in PROBES.items():
        R = [r for r in rows if r["probe"] == p]
        if len({(r["condition"], r["draw"]) for r in R}) < 1 + 16 * DRAWS: print(p, "incomplete, skipped"); continue
        aff = pd.Series({c: np.mean([r["aug_dev_cer"] - r["clean_dev_cer"] for r in R if r["condition"] == c]) for c in L.CONDS})
        own = {g: pd.DataFrame({c: pd.DataFrame([r[g] for r in R if r["condition"] == c])[L.M].mean() for c in L.CONDS}).T for g in ("frame", "line")}
        for g in own: own[g]["Vendi_x_A"] = own[g].Vendi * own[g].A_dist
        y = 100 * np.array([cer[f"{arch}-{c}-s{seed}"] for c in L.CONDS])
        cands = {"affinity": aff.values[:, None], "affinity+img word Vendi+KID+Prec": np.c_[aff.values, img.values]}
        cands.update({f"own {g} {'+'.join(c)}": own[g][list(c)].values for g in own for c in L.COMBOS})
        b0 = L.mae(L.lofo(np.zeros((17, 0)), y), y)
        def ratios(yy):
            b = L.mae(L.lofo(np.zeros((17, 0)), yy), yy)
            return {k: L.mae(L.lofo(X, yy), yy) / b for k, X in cands.items()}
        obs = ratios(y)
        groups = {"affinity": ["affinity"], "own": [k for k in cands if k.startswith("own")], "affinity+img": ["affinity+img word Vendi+KID+Prec"]}
        null = {g: [] for g in groups}
        for _ in range(300):
            yp = y.copy(); yp[L.aug] = rng.permutation(y[L.aug]); rr = ratios(yp)
            for g, ks in groups.items(): null[g].append(min(rr[k] for k in ks))
        res = {"baseline_MAE": b0, "affinity_rho_vs_cer": float(spearmanr(aff.values[L.aug], y[L.aug])[0])}
        for g, ks in groups.items():
            kb = min(ks, key=lambda k: obs[k]); pr = L.lofo(cands[kb], y)
            res[g] = {"best": kb, "ratio": obs[kb], "MAE": obs[kb] * b0, "rho": float(spearmanr(pr[L.aug], y[L.aug])[0]),
                      "perm_median": float(np.median(null[g])), "perm_p": float((np.array(null[g]) <= obs[kb]).mean())}
        out[p] = res
        print(f"\n== {p}: baseline MAE {b0:.2f} | rho(affinity, CER) {res['affinity_rho_vs_cer']:.4f}")
        for g in groups:
            r = res[g]; print(f"   {g:13s} {r['best']:40s} ratio {r['ratio']:.4f} rho {r['rho']:.4f} p {r['perm_p']:.4f}")
    json.dump(out, open("summary.json", "w"), indent=1)
    HfApi().upload_file(path_or_fileobj="summary.json", path_in_repo=f"{PREFIX}/summary.json", repo_id=OUT_REPO, repo_type="dataset",
                        commit_message="model-aware summary")
    print(f"-> {OUT_REPO}/{PREFIX}/summary.json")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--shard", type=int, default=0); ap.add_argument("--nshards", type=int, default=1)
    ap.add_argument("--analyze", action="store_true"); ap.add_argument("--smoke", action="store_true")
    a = ap.parse_args()
    analyze() if a.analyze else work(a.shard, a.nshards, a.smoke)


In [ ]:
%%writefile model_src.json
{
 "htrvt": {
  "cfg": "\nimport os, math\nMODEL = \"htrvt\"\nCONFIG = {\"row_h\": 64, \"max_w\": 2048, \"batch\": 32, \"lr\": 5e-4, \"warmup\": 0.05, \"wd\": 0.05, \"epochs\": 300, \"eval_every\": 10,\n          \"cnn_width\": 256, \"dim\": 384, \"depth\": 6, \"heads\": 6, \"mlp_ratio\": 4, \"dropout\": 0.1, \"mask_ratio\": 0.4, \"mask_span\": 4,\n          \"schedule\": \"linear warm-up + cosine decay to 1% of the peak\"}\nROW_H, MAX_W, BATCH = CONFIG[\"row_h\"], CONFIG[\"max_w\"], CONFIG[\"batch\"]\nLR, WARMUP, WD, EPOCHS, EVAL_EVERY = CONFIG[\"lr\"], CONFIG[\"warmup\"], CONFIG[\"wd\"], CONFIG[\"epochs\"], CONFIG[\"eval_every\"]\nNUM_WORKERS, EVAL_BS = min(8, os.cpu_count()), 32\ndef schedule(step, total, warm):\n    if step < warm: return (step + 1) / warm\n    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))\n",
  "fmt": "\nSTRIDE = 4   # the CNN reduces width 4x -> one CTC step per 4 px\ndef fmt(g):\n    \"\"\"64 px gray line -> mirror (RTL -> LTR) -> width capped at MAX_W -> (1,64,W) in [-1,1] (ink dark) + CTC length\"\"\"\n    g = g[:, ::-1]\n    if g.shape[1] > MAX_W: g = A.cv2.resize(np.ascontiguousarray(g), (MAX_W, ROW_H), interpolation=A.cv2.INTER_AREA)\n    t = torch.from_numpy(np.ascontiguousarray(g)).float().div(255).sub(0.5).div(0.5).unsqueeze(0)\n    return t, max(1, t.shape[-1] // STRIDE)\ndef collate(b):\n    W = max(x[0].shape[-1] for x in b); W = int(math.ceil(W / STRIDE) * STRIDE)\n    px = torch.ones(len(b), 1, ROW_H, W)   # pad with white (+1)\n    for i, x in enumerate(b): px[i, :, :, :x[0].shape[-1]] = x[0]\n    return px, torch.tensor([x[1] for x in b]), [x[2] for x in b], [x[3] for x in b]\n",
  "model": "\nclass ResBlock(nn.Module):\n    def __init__(self, cin, cout, pool):\n        super().__init__()\n        self.c1, self.b1 = nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout)\n        self.c2, self.b2 = nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout)\n        self.sc = nn.Identity() if cin == cout else nn.Sequential(nn.Conv2d(cin, cout, 1, bias=False), nn.BatchNorm2d(cout))\n        self.pool = nn.MaxPool2d(pool) if pool != (1, 1) else nn.Identity()\n    def forward(self, x):\n        y = self.b2(self.c2(F.relu(self.b1(self.c1(x)))))\n        return self.pool(F.relu(y + self.sc(x)))\n\nclass HTRVT(nn.Module):\n    \"\"\"Our implementation after HTR-VT (Li et al., Pattern Recognition 2025): a residual CNN turns the 64 px line into a feature\n    sequence (height 64 -> 4, folded into channels; width / 4), span masking on that sequence during training, sinusoidal positions,\n    a pre-norm transformer encoder, a linear CTC head. Trained from scratch.\"\"\"\n    def __init__(self, n_classes, cnn_width, dim, depth, heads, mlp_ratio, dropout, mask_ratio, mask_span, **_):\n        super().__init__()\n        w1, w2, w3 = cnn_width // 4, cnn_width // 2, cnn_width\n        self.cnn = nn.Sequential(nn.Conv2d(1, w1, 3, padding=1, bias=False), nn.BatchNorm2d(w1), nn.ReLU(),\n                                 ResBlock(w1, w1, (2, 2)), ResBlock(w1, w2, (2, 2)), ResBlock(w2, w2, (2, 1)),\n                                 ResBlock(w2, w3, (2, 1)), ResBlock(w3, w3, (1, 1)))\n        self.proj = nn.Linear(w3 * 4, dim); self.mask_ratio, self.mask_span = mask_ratio, mask_span\n        self.drop = nn.Dropout(dropout)\n        layer = nn.TransformerEncoderLayer(dim, heads, int(dim * mlp_ratio), dropout, activation=\"gelu\", batch_first=True, norm_first=True)\n        self.enc = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)\n        self.norm, self.head = nn.LayerNorm(dim), nn.Linear(dim, n_classes)\n        pe = torch.zeros(4096, dim); pos = torch.arange(4096).float()[:, None]\n        div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))\n        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div); self.register_buffer(\"pe\", pe, persistent=False)\n        for m in self.modules():\n            if isinstance(m, nn.Linear): nn.init.trunc_normal_(m.weight, std=0.02); nn.init.zeros_(m.bias) if m.bias is not None else None\n    def span_mask(self, x):\n        B, T, _ = x.shape; n = max(1, int(T * self.mask_ratio / self.mask_span))\n        st = torch.randint(0, max(1, T - self.mask_span), (B, n), device=x.device); m = torch.zeros(B, T, dtype=torch.bool, device=x.device)\n        for o in range(self.mask_span): m.scatter_(1, (st + o).clamp(max=T - 1), True)\n        return x.masked_fill(m[..., None], 0.0)\n    def forward(self, px, lens=None):   # lens = CTC steps per line; padded steps are masked out of attention\n        f = self.cnn(px); B, Cc, Hh, W = f.shape\n        x = self.proj(f.permute(0, 3, 1, 2).reshape(B, W, Cc * Hh))\n        if self.training and self.mask_ratio > 0: x = self.span_mask(x)\n        pad = None if lens is None else torch.arange(W, device=x.device)[None, :] >= lens[:, None]\n        x = self.enc(self.drop(x + self.pe[:W]), src_key_padding_mask=pad)\n        return F.log_softmax(self.head(self.norm(x)).float(), dim=-1)\n\ntorch.manual_seed(SEED)\nmodel = HTRVT(N_CLASSES, **CONFIG).to(device)\nPARAM_GROUPS = [{\"params\": list(model.parameters()), \"lr\": LR}]\nprint(f\"HTR-VT (ours) {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters, trained from scratch\")\n"
 },
 "trocrctc": {
  "cfg": "\nimport os, math\nMODEL = \"trocrctc\"\nCONFIG = {\"enc\": \"microsoft/trocr-base-handwritten\", \"row_h\": 64, \"sub\": 2, \"batch\": 32, \"lr_enc\": 5e-05, \"lr_head\": 3e-4, \"warmup\": 0.05,\n          \"wd\": 0.01, \"epochs\": 40, \"eval_every\": 5, \"schedule\": \"linear warm-up + linear decay\"}\nENC_SRC, ROW_H, SUB, BATCH = CONFIG[\"enc\"], CONFIG[\"row_h\"], CONFIG[\"sub\"], CONFIG[\"batch\"]\nLR_ENC, LR_HEAD, WARMUP, WD, EPOCHS, EVAL_EVERY = CONFIG[\"lr_enc\"], CONFIG[\"lr_head\"], CONFIG[\"warmup\"], CONFIG[\"wd\"], CONFIG[\"epochs\"], CONFIG[\"eval_every\"]\nNUM_WORKERS, EVAL_BS = min(8, os.cpu_count()), 64\ndef schedule(step, total, warm): return min(1.0, (step + 1) / warm) * max(0.0, (total - step) / max(1, total - warm))\n",
  "fmt": "\nC, P = 384, 16; RROWS = C // ROW_H; MAX_COLS = RROWS * (C // P)\ndef fmt(g):\n    \"\"\"64 px gray line -> mirror (RTL -> LTR) -> wrapped into the 384x384 ViT square in rows of 384 px -> (3,384,384) in [-1,1] + CTC length\"\"\"\n    g = g[:, ::-1]; nw = g.shape[1]; cont = np.full((C, C), 255, np.uint8); sx, r = 0, 0\n    while sx < nw and r < RROWS:\n        seg = g[:, sx:sx + C]; cont[r * ROW_H:(r + 1) * ROW_H, :seg.shape[1]] = seg; sx += seg.shape[1]; r += 1\n    t = torch.from_numpy(cont).float().div(255).sub(0.5).div(0.5).unsqueeze(0).expand(3, C, C).contiguous()\n    return t, min(MAX_COLS, math.ceil(min(nw, RROWS * C) / P)) * SUB\ndef collate(b):\n    return torch.stack([x[0] for x in b]), torch.tensor([x[1] for x in b]), [x[2] for x in b], [x[3] for x in b]\n",
  "model": "\nfrom transformers import VisionEncoderDecoderModel\nclass EncoderLinearCTC(nn.Module):\n    \"\"\"ViT patch tokens (24x24) -> reading-order frames (one per 16 px column, the ROW_H//16 patch rows of a text row concatenated)\n    -> ONE Linear giving SUB CTC steps per column (exp38 / Option-B recipe).\"\"\"\n    def __init__(self, vit):\n        super().__init__(); self.vit = vit\n        if getattr(self.vit, \"pooler\", None) is not None: self.vit.pooler = None\n        self.Hd = vit.config.hidden_size; self.k = ROW_H // 16\n        self.classifier = nn.Linear(self.k * self.Hd, SUB * N_CLASSES)\n        nn.init.trunc_normal_(self.classifier.weight, std=0.02); nn.init.zeros_(self.classifier.bias)\n    def forward(self, px, lens=None):   # lens unused: every input is a full 384x384 square\n        tok = self.vit(pixel_values=px).last_hidden_state[:, -576:]; B = tok.shape[0]   # the 24x24 patch tokens (ViT: after CLS; DeiT: after CLS + distillation)\n        g = tok.reshape(B, 24, 24, self.Hd)[:, :RROWS * self.k]\n        g = g.reshape(B, RROWS, self.k, 24, self.Hd).permute(0, 1, 3, 2, 4).reshape(B, RROWS * 24, self.k * self.Hd)\n        return F.log_softmax(self.classifier(g).reshape(B, RROWS * 24 * SUB, N_CLASSES).float(), dim=-1)\ntorch.manual_seed(SEED)\nmodel = EncoderLinearCTC(VisionEncoderDecoderModel.from_pretrained(ENC_SRC).encoder).to(device)\nPARAM_GROUPS = [{\"params\": list(model.vit.parameters()), \"lr\": LR_ENC}, {\"params\": list(model.classifier.parameters()), \"lr\": LR_HEAD}]\nprint(f\"encoder {sum(p.numel() for p in model.vit.parameters())/1e6:.1f}M | classifier {sum(p.numel() for p in model.classifier.parameters())/1e6:.2f}M\")\n"
 },
 "trocrsmall": {
  "cfg": "\nimport os, math\nMODEL = \"trocrsmall\"\nCONFIG = {\"enc\": \"microsoft/trocr-small-stage1\", \"row_h\": 64, \"sub\": 2, \"batch\": 32, \"lr_enc\": 5e-05, \"lr_head\": 3e-4, \"warmup\": 0.05,\n          \"wd\": 0.01, \"epochs\": 40, \"eval_every\": 5, \"schedule\": \"linear warm-up + linear decay\"}\nENC_SRC, ROW_H, SUB, BATCH = CONFIG[\"enc\"], CONFIG[\"row_h\"], CONFIG[\"sub\"], CONFIG[\"batch\"]\nLR_ENC, LR_HEAD, WARMUP, WD, EPOCHS, EVAL_EVERY = CONFIG[\"lr_enc\"], CONFIG[\"lr_head\"], CONFIG[\"warmup\"], CONFIG[\"wd\"], CONFIG[\"epochs\"], CONFIG[\"eval_every\"]\nNUM_WORKERS, EVAL_BS = min(8, os.cpu_count()), 64\ndef schedule(step, total, warm): return min(1.0, (step + 1) / warm) * max(0.0, (total - step) / max(1, total - warm))\n",
  "fmt": "\nC, P = 384, 16; RROWS = C // ROW_H; MAX_COLS = RROWS * (C // P)\ndef fmt(g):\n    \"\"\"64 px gray line -> mirror (RTL -> LTR) -> wrapped into the 384x384 ViT square in rows of 384 px -> (3,384,384) in [-1,1] + CTC length\"\"\"\n    g = g[:, ::-1]; nw = g.shape[1]; cont = np.full((C, C), 255, np.uint8); sx, r = 0, 0\n    while sx < nw and r < RROWS:\n        seg = g[:, sx:sx + C]; cont[r * ROW_H:(r + 1) * ROW_H, :seg.shape[1]] = seg; sx += seg.shape[1]; r += 1\n    t = torch.from_numpy(cont).float().div(255).sub(0.5).div(0.5).unsqueeze(0).expand(3, C, C).contiguous()\n    return t, min(MAX_COLS, math.ceil(min(nw, RROWS * C) / P)) * SUB\ndef collate(b):\n    return torch.stack([x[0] for x in b]), torch.tensor([x[1] for x in b]), [x[2] for x in b], [x[3] for x in b]\n",
  "model": "\nfrom transformers import VisionEncoderDecoderModel\nclass EncoderLinearCTC(nn.Module):\n    \"\"\"ViT patch tokens (24x24) -> reading-order frames (one per 16 px column, the ROW_H//16 patch rows of a text row concatenated)\n    -> ONE Linear giving SUB CTC steps per column (exp38 / Option-B recipe).\"\"\"\n    def __init__(self, vit):\n        super().__init__(); self.vit = vit\n        if getattr(self.vit, \"pooler\", None) is not None: self.vit.pooler = None\n        self.Hd = vit.config.hidden_size; self.k = ROW_H // 16\n        self.classifier = nn.Linear(self.k * self.Hd, SUB * N_CLASSES)\n        nn.init.trunc_normal_(self.classifier.weight, std=0.02); nn.init.zeros_(self.classifier.bias)\n    def forward(self, px, lens=None):   # lens unused: every input is a full 384x384 square\n        tok = self.vit(pixel_values=px).last_hidden_state[:, -576:]; B = tok.shape[0]   # the 24x24 patch tokens (ViT: after CLS; DeiT: after CLS + distillation)\n        g = tok.reshape(B, 24, 24, self.Hd)[:, :RROWS * self.k]\n        g = g.reshape(B, RROWS, self.k, 24, self.Hd).permute(0, 1, 3, 2, 4).reshape(B, RROWS * 24, self.k * self.Hd)\n        return F.log_softmax(self.classifier(g).reshape(B, RROWS * 24 * SUB, N_CLASSES).float(), dim=-1)\ntorch.manual_seed(SEED)\nmodel = EncoderLinearCTC(VisionEncoderDecoderModel.from_pretrained(ENC_SRC).encoder).to(device)\nPARAM_GROUPS = [{\"params\": list(model.vit.parameters()), \"lr\": LR_ENC}, {\"params\": list(model.classifier.parameters()), \"lr\": LR_HEAD}]\nprint(f\"encoder {sum(p.numel() for p in model.vit.parameters())/1e6:.1f}M | classifier {sum(p.numel() for p in model.classifier.parameters())/1e6:.2f}M\")\n"
 },
 "trocrscratch": {
  "cfg": "\nimport os, math\nMODEL = \"trocrscratch\"\nCONFIG = {\"enc_config\": \"microsoft/trocr-small-stage1\", \"init\": \"random\", \"row_h\": 64, \"sub\": 2, \"batch\": 32, \"lr\": 5e-4, \"warmup\": 0.05,\n          \"wd\": 0.05, \"epochs\": 300, \"eval_every\": 10, \"schedule\": \"linear warm-up + cosine decay to 1% of the peak (as HTR-VT)\"}\nENC_CFG_SRC, ROW_H, SUB, BATCH = CONFIG[\"enc_config\"], CONFIG[\"row_h\"], CONFIG[\"sub\"], CONFIG[\"batch\"]\nLR, WARMUP, WD, EPOCHS, EVAL_EVERY = CONFIG[\"lr\"], CONFIG[\"warmup\"], CONFIG[\"wd\"], CONFIG[\"epochs\"], CONFIG[\"eval_every\"]\nNUM_WORKERS, EVAL_BS = min(8, os.cpu_count()), 64\ndef schedule(step, total, warm):\n    if step < warm: return (step + 1) / warm\n    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))\n",
  "fmt": "\nC, P = 384, 16; RROWS = C // ROW_H; MAX_COLS = RROWS * (C // P)\ndef fmt(g):\n    \"\"\"64 px gray line -> mirror (RTL -> LTR) -> wrapped into the 384x384 ViT square in rows of 384 px -> (3,384,384) in [-1,1] + CTC length\"\"\"\n    g = g[:, ::-1]; nw = g.shape[1]; cont = np.full((C, C), 255, np.uint8); sx, r = 0, 0\n    while sx < nw and r < RROWS:\n        seg = g[:, sx:sx + C]; cont[r * ROW_H:(r + 1) * ROW_H, :seg.shape[1]] = seg; sx += seg.shape[1]; r += 1\n    t = torch.from_numpy(cont).float().div(255).sub(0.5).div(0.5).unsqueeze(0).expand(3, C, C).contiguous()\n    return t, min(MAX_COLS, math.ceil(min(nw, RROWS * C) / P)) * SUB\ndef collate(b):\n    return torch.stack([x[0] for x in b]), torch.tensor([x[1] for x in b]), [x[2] for x in b], [x[3] for x in b]\n",
  "model": "\nfrom transformers import AutoModel, VisionEncoderDecoderConfig\nclass EncoderLinearCTC(nn.Module):\n    \"\"\"ViT patch tokens (24x24) -> reading-order frames (one per 16 px column, the ROW_H//16 patch rows of a text row concatenated)\n    -> ONE Linear giving SUB CTC steps per column (exp38 / Option-B recipe).\"\"\"\n    def __init__(self, vit):\n        super().__init__(); self.vit = vit\n        if getattr(self.vit, \"pooler\", None) is not None: self.vit.pooler = None\n        self.Hd = vit.config.hidden_size; self.k = ROW_H // 16\n        self.classifier = nn.Linear(self.k * self.Hd, SUB * N_CLASSES)\n        nn.init.trunc_normal_(self.classifier.weight, std=0.02); nn.init.zeros_(self.classifier.bias)\n    def forward(self, px, lens=None):   # lens unused: every input is a full 384x384 square\n        tok = self.vit(pixel_values=px).last_hidden_state[:, -576:]; B = tok.shape[0]   # the 24x24 patch tokens (ViT: after CLS; DeiT: after CLS + distillation)\n        g = tok.reshape(B, 24, 24, self.Hd)[:, :RROWS * self.k]\n        g = g.reshape(B, RROWS, self.k, 24, self.Hd).permute(0, 1, 3, 2, 4).reshape(B, RROWS * 24, self.k * self.Hd)\n        return F.log_softmax(self.classifier(g).reshape(B, RROWS * 24 * SUB, N_CLASSES).float(), dim=-1)\ntorch.manual_seed(SEED)\nenc_cfg = VisionEncoderDecoderConfig.from_pretrained(ENC_CFG_SRC).encoder   # architecture only: NO pretrained weights\nmodel = EncoderLinearCTC(AutoModel.from_config(enc_cfg)).to(device)\nprint(\"random init:\", type(model.vit).__name__, enc_cfg.hidden_size, \"dim,\", enc_cfg.num_hidden_layers, \"layers\")\nPARAM_GROUPS = [{\"params\": list(model.parameters()), \"lr\": LR}]\nprint(f\"encoder {sum(p.numel() for p in model.vit.parameters())/1e6:.1f}M | classifier {sum(p.numel() for p in model.classifier.parameters())/1e6:.2f}M\")\n"
 }
}

## Run: one worker per GPU (logs in `worker<i>.log`)

`SMOKE = True` runs one worker on a few lines without uploading.

In [ ]:
import subprocess, os, time
SMOKE = False
n = 1 if SMOKE else 2
procs = [subprocess.Popen(f'python model_aware.py --shard {i} --nshards {n}' + (' --smoke' if SMOKE else '') + f' > worker{i}.log 2>&1',
                          shell=True, env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(i)}) for i in range(n)]
seen = [0] * n
while any(p.poll() is None for p in procs):
    time.sleep(30)
    for i in range(n):
        lines = [l for l in open(f'worker{i}.log', errors='ignore').read().splitlines() if l.startswith(('[', 'train', 'shard', 'upload'))]
        for l in lines[seen[i]:]: print(f'w{i}', l)
        seen[i] = len(lines)
print('exit codes', [p.returncode for p in procs])
for i in range(n):
    if procs[i].returncode: print(f'--- worker{i}.log tail ---'); print(open(f'worker{i}.log').read()[-3000:])

## Leave-one-family-out: affinity / own-feature metrics / affinity + image metrics -> `model_aware/summary.json`

In [ ]:
!python model_aware.py --analyze